# Spot displacement vs wavefront gradient

In geometric optics the transverse ray aberration at the focal plane is
proportional to the wavefront slope at the ray's pupil coordinate:

$$
\delta x \propto -\frac{\partial W}{\partial p_x}, \qquad
\delta y \propto -\frac{\partial W}{\partial p_y}
$$

where $W(p_x, p_y)$ is the wavefront (OPD) in the pupil, and
$(\delta x, \delta y)$ are the spot displacements.

More precisely, for a focal length $f$ and wavelength $\lambda$, the
proportionality constant is $-f / R_{\text{pupil}}$ (with some sign and
unit conventions).

The goal of this notebook is to verify this relationship numerically
using raytraced spots and Zernike coefficients from
`RaytracedOpticalModel`, so that we can later use the wavefront gradient
to *predict* spot sensitivities from Zernike sensitivities without
additional ray tracing.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import batoid
from batoid_rubin import LSSTBuilder
import astropy.units as u
from astropy.coordinates import Angle
from lsst.obs.lsst import LsstCam
import galsim
import matplotlib.pyplot as plt

from StarSharp import RaytracedOpticalModel, FieldCoords, State

## Setup

In [ ]:
fiducial = batoid.Optic.fromYaml("Rubin_v3.14_r.yaml")
wavelength = 620 * u.nm
rtp = Angle("0 deg")
camera = LsstCam().getCamera()

builder = LSSTBuilder(
    fiducial,
    dof_coord_system="OCS",
    flip_m2_bending_modes=False,
    dof_angle_units="degree",
)
model = RaytracedOpticalModel(
    builder=builder,
    rtp=rtp,
    wavelength=wavelength,
    camera=camera,
)

## Compute spots and Zernikes at a few field points

We'll use a single perturbed state (e.g. camera dz) and a handful of
field points so the raytracing is fast.

In [ ]:
# A small perturbation: 10 µm of camera dz (DOF index 5)
dof = np.zeros(50)
dof[5] = 10.0  # µm
state = State(dof, basis="f")

# A few field points across the field of view
# field = model.make_hex_field(outer=1.5 * u.deg, nrad=3)
field = model.make_ccd_field(nx=1, detnums=[94])

In [ ]:
# Compute spots (with pupil coordinates) and Zernikes
nrad = 10
spots = model.spots(field=field, state=state, nrad=nrad, reference="ring")
zk = model.zernikes(field=field, state=state, jmax=66, rings=20, algorithm="ta")

print(f"nfield = {spots.nfield}, nray = {spots.nray}")
print(f"jmax = {zk.jmax}")
print(f"px shape = {spots.px.shape}")

## Evaluate the Zernike wavefront gradient at the pupil coordinates

The Zernike coefficients describe the wavefront $W(p_x, p_y)$ in the
pupil.  We use `galsim.zernike.Zernike` to evaluate the gradient
$\nabla W$ at the exact pupil positions used for raytracing.

In [ ]:
# Pupil coordinates used by the raytracer (in metres)
px = spots.px.to_value(u.m)
py = spots.py.to_value(u.m)

R_outer = zk.R_outer.to_value(u.m)
R_inner = zk.R_inner.to_value(u.m)

# Zernikes are in OCS; spots are in CCS. Work in OCS for both.
zk_ocs = zk.ocs
spots_ocs = spots.ocs

# For each field point, build a galsim Zernike and evaluate gradients
grad_x_all = []  # dW/dpx at each ray, for each field point
grad_y_all = []  # dW/dpy

for i in range(zk_ocs.nfield):
    z = galsim.zernike.Zernike(
        zk_ocs.coefs[i].to_value(u.um),
        R_outer=R_outer,
        R_inner=R_inner,
    )
    gradx_z, grady_z = z.gradX, z.gradY  # derivative Zernike objects
    grad_x_all.append(gradx_z(px, py))   # µm / m  (wavefront gradient)
    grad_y_all.append(grady_z(px, py))

grad_x = np.array(grad_x_all)  # (nfield, nray) in µm/m
grad_y = np.array(grad_y_all)

## Compare spot displacements to wavefront gradients

We expect $\delta x \propto -\partial W / \partial p_x$ with a single
constant of proportionality (the effective focal length divided by the
pupil radius, roughly).  Let's scatter-plot them and fit the slope.

In [ ]:
# Spot displacements in OCS, in µm
dx = spots_ocs.dx.to_value(u.um)  # (nfield, nray)
dy = spots_ocs.dy.to_value(u.um)
vig = spots_ocs.vignetted

# Mask vignetted rays
good = ~vig

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- x component ---
ax = axes[0]
gx = -grad_x[good]  # negative because transverse aberration = -dW/dp
sx = dx[good]
ax.scatter(gx, sx, s=1, alpha=0.3)
# Fit proportionality constant
# slope_x = np.dot(gx, sx) / np.dot(gx, gx)
slope_x = 10.31
xlim = np.array([gx.min(), gx.max()])
ax.plot(xlim, slope_x * xlim, 'r-', lw=2, label=f'slope = {slope_x:.4f} m')
ax.set_xlabel(r'$-\partial W / \partial p_x$  [µm / m]')
ax.set_ylabel(r'Spot $\delta x$  [µm]')
ax.set_title('x component')
ax.legend()

# --- y component ---
ax = axes[1]
gy = -grad_y[good]
sy = dy[good]
ax.scatter(gy, sy, s=1, alpha=0.3)
# slope_y = np.dot(gy, sy) / np.dot(gy, gy)
slope_y = 10.31
ylim = np.array([gy.min(), gy.max()])
ax.plot(ylim, slope_y * ylim, 'r-', lw=2, label=f'slope = {slope_y:.4f} m')
ax.set_xlabel(r'$-\partial W / \partial p_y$  [µm / m]')
ax.set_ylabel(r'Spot $\delta y$  [µm]')
ax.set_title('y component')
ax.legend()

fig.suptitle('Spot displacement vs wavefront gradient', fontsize=14)
fig.tight_layout()
plt.show()

print(f"Fitted proportionality constant (x): {slope_x:.6f} m")
print(f"Fitted proportionality constant (y): {slope_y:.6f} m")
print(f"Ratio slope_x / slope_y: {slope_x / slope_y:.6f}")

## Residuals

How well does the linear relationship hold?  Check the RMS residual
relative to the spot RMS.

In [ ]:
pred_x = slope_x * (-grad_x)
pred_y = slope_y * (-grad_y)

res_x = (dx - pred_x)[good]
res_y = (dy - pred_y)[good]

rms_dx = np.sqrt(np.mean(dx[good]**2))
rms_dy = np.sqrt(np.mean(dy[good]**2))
rms_res_x = np.sqrt(np.mean(res_x**2))
rms_res_y = np.sqrt(np.mean(res_y**2))

print(f"Spot RMS  x: {rms_dx:.4f} µm,  y: {rms_dy:.4f} µm")
print(f"Resid RMS x: {rms_res_x:.4f} µm,  y: {rms_res_y:.4f} µm")
print(f"Fractional x: {rms_res_x / rms_dx:.2e},  y: {rms_res_y / rms_dy:.2e}")

## Check the sensitivity (differential) version

The above compares *absolute* spots to the wavefront gradient.  For the
`LinearOpticalModel` use case what matters is whether the *change* in
spots due to a DOF perturbation is proportional to the *change* in the
wavefront gradient.  Compute the finite-difference spot and Zernike
sensitivities and compare them.

In [ ]:
# Use a smaller field for sensitivity (it's slow)
# sens_field = model.make_hex_field(outer=1.75 * u.deg, nrad=5)
# sens_field = model.make_ccd_field(nx=1, detnums=np.arange(4, 189, 9))
sens_field = model.make_ccd_field(nx=1, detnums=[4+9*3])

# # Choose a few DOFs to perturb
# use_dof = np.array([0, 2, 3, 5, 8, 9, 11, 15, 31, 36])
# step_vals = np.array([10.0, 100.0, 0.001, 10.0, 100.0, 0.001, 0.1, 0.1, 0.1, 0.1])
# steps = State(
#     value=step_vals,
#     basis="x",
#     use_dof=use_dof,
#     n_dof=50,
# )
steps = State(
    value=model.steps,
    basis="f",
)
use_dof = list(range(50))

print("Computing Zernike sensitivity...")
zk_sens = model.zernikes_sensitivity(sens_field, steps, rings=20, jmax=78, algorithm="ta")
print("Computing spot sensitivity...")
sp_sens = model.spots_sensitivity(sens_field, steps, nrad=nrad, reference="ring")

In [ ]:
# Get the pupil coords from the nominal spots
px_s = sp_sens.nominal.px.to_value(u.m)
py_s = sp_sens.nominal.py.to_value(u.m)

# Convert spot gradient to OCS (Zernike gradients are in OCS)
sp_grad_ocs = sp_sens.gradient.ocs

# For each DOF, compare d(spot)/d(dof) to d(gradW)/d(dof)
ndof = len(use_dof)

fig, axes = plt.subplots(ndof, 2, figsize=(12, 4 * ndof), squeeze=False)

for idof in range(ndof):
    # Spot sensitivity: (nfield, nray) in µm
    d_dx = sp_grad_ocs.dx[idof].to_value(u.um)  # (nfield, nray)
    d_dy = sp_grad_ocs.dy[idof].to_value(u.um)
    vig_s = sp_grad_ocs.vignetted[idof]
    good_s = ~vig_s

    # Zernike sensitivity: (nfield, jmax+1) in µm
    zk_grad_coefs = zk_sens.gradient.coefs[idof]  # (nfield, jmax+1)
    jmax_s = zk_grad_coefs.shape[-1] - 1

    # Evaluate wavefront gradient at pupil coords for each field point
    dgrad_x = np.zeros_like(d_dx)
    dgrad_y = np.zeros_like(d_dy)
    for ifield in range(zk_grad_coefs.shape[0]):
        coef = zk_grad_coefs[ifield].to_value(u.um)
        coef[:4] = 0.0
        z = galsim.zernike.Zernike(
            coef,
            R_outer=R_outer,
            R_inner=R_inner,
        )
        dgrad_x[ifield] = z.gradX(px_s, py_s)
        dgrad_y[ifield] = z.gradY(px_s, py_s)

    # Plot x component
    ax = axes[idof, 0]
    gx_s = -dgrad_x[good_s]
    sx_s = d_dx[good_s]
    # slope = np.dot(gx_s, sx_s) / np.dot(gx_s, gx_s) if np.any(gx_s != 0) else 0
    slope = 10.31
    ax.scatter(gx_s, sx_s, s=1, alpha=0.4)
    if np.any(gx_s != 0):
        xl = np.array([gx_s.min(), gx_s.max()])
        ax.plot(xl, slope * xl, 'r-', lw=2, label=f'slope={slope:.4f}')
    ax.set_xlabel(r'$-\partial (\delta W) / \partial p_x$')
    ax.set_ylabel(r'$\delta(\mathrm{spot}_x)$  [µm]')
    ax.axhline(0, color='w', lw=0.5, ls='--')
    ax.axvline(0, color='w', lw=0.5, ls='--')
    ax.set_title(f'DOF {use_dof[idof]} — x')
    ax.legend(fontsize=8)

    # Plot y component
    ax = axes[idof, 1]
    gy_s = -dgrad_y[good_s]
    sy_s = d_dy[good_s]
    # slope = np.dot(gy_s, sy_s) / np.dot(gy_s, gy_s) if np.any(gy_s != 0) else 0
    slope = 10.31
    ax.scatter(gy_s, sy_s, s=1, alpha=0.4)
    if np.any(gy_s != 0):
        yl = np.array([gy_s.min(), gy_s.max()])
        ax.plot(yl, slope * yl, 'r-', lw=2, label=f'slope={slope:.4f}')
    ax.set_xlabel(r'$-\partial (\delta W) / \partial p_y$')
    ax.set_ylabel(r'$\delta(\mathrm{spot}_y)$  [µm]')
    ax.axhline(0, color='w', lw=0.5, ls='--')
    ax.axvline(0, color='w', lw=0.5, ls='--')
    ax.set_title(f'DOF {use_dof[idof]} — y')
    ax.legend(fontsize=8)

fig.suptitle('Spot sensitivity vs wavefront-gradient sensitivity', fontsize=14)
fig.tight_layout()
plt.show()

## Quantitative summary

For each DOF, compute the fitted proportionality constant and the
fractional residual RMS.

In [ ]:
print(f"{'DOF':>4s}  {'slope_x':>10s}  {'slope_y':>10s}  {'frac_res_x':>10s}  {'frac_res_y':>10s}")
print("-" * 52)

for idof in range(ndof):
    d_dx = sp_grad_ocs.dx[idof].to_value(u.um)  # (nfield, nray)
    d_dy = sp_grad_ocs.dy[idof].to_value(u.um)
    vig_s = sp_grad_ocs.vignetted[idof]
    good_s = ~vig_s

    zk_grad_coefs = zk_sens.gradient.coefs[idof]
    dgrad_x = np.zeros_like(d_dx)
    dgrad_y = np.zeros_like(d_dy)
    for ifield in range(zk_grad_coefs.shape[0]):
        z = galsim.zernike.Zernike(
            zk_grad_coefs[ifield].to_value(u.um),
            R_outer=R_outer,
            R_inner=R_inner,
        )
        dgrad_x[ifield] = z.gradX(px_s, py_s)
        dgrad_y[ifield] = z.gradY(px_s, py_s)

    gx = -dgrad_x[good_s]
    gy = -dgrad_y[good_s]
    sx = d_dx[good_s]
    sy = d_dy[good_s]

    sl_x = np.dot(gx, sx) / np.dot(gx, gx) if np.any(gx != 0) else np.nan
    sl_y = np.dot(gy, sy) / np.dot(gy, gy) if np.any(gy != 0) else np.nan

    res_x = sx - sl_x * gx
    res_y = sy - sl_y * gy
    frac_x = np.sqrt(np.mean(res_x**2)) / np.sqrt(np.mean(sx**2)) if np.any(sx != 0) else np.nan
    frac_y = np.sqrt(np.mean(res_y**2)) / np.sqrt(np.mean(sy**2)) if np.any(sy != 0) else np.nan

    print(f"{use_dof[idof]:4d}  {sl_x:10.4f}  {sl_y:10.4f}  {frac_x:10.2e}  {frac_y:10.2e}")

## Cross-component check

Is a single scalar proportionality constant enough, or do we need a
2×2 matrix?  Plot $\delta x$ vs $-\partial W / \partial p_y$ (the
"wrong" component) — it should be near zero if a scalar suffices.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Use DOF 0 as an example
idof = 0
d_dx = sp_grad_ocs.dx[idof].to_value(u.um)  # (nfield, nray)
d_dy = sp_grad_ocs.dy[idof].to_value(u.um)
vig_s = sp_grad_ocs.vignetted[idof]
good_s = ~vig_s

zk_grad_coefs = zk_sens.gradient.coefs[idof]
dgrad_x = np.zeros_like(d_dx)
dgrad_y = np.zeros_like(d_dy)
for ifield in range(zk_grad_coefs.shape[0]):
    z = galsim.zernike.Zernike(
        zk_grad_coefs[ifield].to_value(u.um),
        R_outer=R_outer,
        R_inner=R_inner,
    )
    dgrad_x[ifield] = z.gradX(px_s, py_s)
    dgrad_y[ifield] = z.gradY(px_s, py_s)

# Cross terms
ax = axes[0]
ax.scatter(-dgrad_y[good_s], d_dx[good_s], s=1, alpha=0.3)
ax.set_xlabel(r'$-\partial W / \partial p_y$')
ax.set_ylabel(r'$\delta x$  [µm]')
ax.set_title(f'DOF {use_dof[idof]}: cross-term x vs grad_y')

ax = axes[1]
ax.scatter(-dgrad_x[good_s], d_dy[good_s], s=1, alpha=0.3)
ax.set_xlabel(r'$-\partial W / \partial p_x$')
ax.set_ylabel(r'$\delta y$  [µm]')
ax.set_title(f'DOF {use_dof[idof]}: cross-term y vs grad_x')

fig.tight_layout()
plt.show()

## Conclusion

If the scatter plots above show tight linear relationships with small
residuals and negligible cross-terms, then we can reliably predict
spots from the Zernike wavefront gradient.  This means
`LinearOpticalModel` can derive spot sensitivities from a
`Sensitivity[DoubleZernikes]` without needing separate raytraced spot
sensitivities — we just need the pupil coordinates and the
proportionality constant.